In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("S10-Validacion-Dataset-Historico")
    .master("local[2]")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

RUTA = "/workspace/artifacts/s10/parquet_zenodo_trayectorias"

df = spark.read.parquet(RUTA)

print("TOTAL REGISTROS:", df.count())
print("TRAYECTORIAS:", df.select("trayectoriaId").distinct().count())
print("DIAS:", df.select("fecha").distinct().count())

df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 13:29:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

TOTAL REGISTROS: 11251127


TRAYECTORIAS: 2465


[Stage 11:======================================================> (48 + 1) / 49]

DIAS: 364
root
 |-- trayectoriaId: string (nullable = true)
 |-- epochSeconds: long (nullable = true)
 |-- eventTimestamp: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- geoAltitude: double (nullable = true)
 |-- velocity: double (nullable = true)
 |-- verticalRate: double (nullable = true)
 |-- trueTrack: double (nullable = true)
 |-- callsign: string (nullable = true)
 |-- sourceDataset: string (nullable = true)
 |-- rutaArchivo: string (nullable = true)
 |-- fecha: date (nullable = true)



In [2]:
ventanas = (
    df.groupBy(
        F.window("eventTimestamp", "5 minutes")
    )
    .agg(
        F.countDistinct("trayectoriaId").alias("aeronaves"),
        F.count("*").alias("observaciones")
    )
    .select(
        F.col("window.start").alias("inicio"),
        F.col("window.end").alias("fin"),
        "aeronaves",
        "observaciones"
    )
)

print("VENTANAS CON DATOS:", ventanas.count())

ventanas.orderBy("inicio").show(20, False)

VENTANAS CON DATOS: 38848


[Stage 23:======================================================> (48 + 1) / 49]

+-------------------+-------------------+---------+-------------+
|inicio             |fin                |aeronaves|observaciones|
+-------------------+-------------------+---------+-------------+
|2025-01-01 06:10:00|2025-01-01 06:15:00|1        |153          |
|2025-01-01 06:15:00|2025-01-01 06:20:00|1        |300          |
|2025-01-01 06:20:00|2025-01-01 06:25:00|1        |300          |
|2025-01-01 06:25:00|2025-01-01 06:30:00|1        |300          |
|2025-01-01 06:30:00|2025-01-01 06:35:00|1        |300          |
|2025-01-01 06:35:00|2025-01-01 06:40:00|1        |300          |
|2025-01-01 06:40:00|2025-01-01 06:45:00|1        |300          |
|2025-01-01 06:45:00|2025-01-01 06:50:00|1        |300          |
|2025-01-01 06:50:00|2025-01-01 06:55:00|1        |300          |
|2025-01-01 06:55:00|2025-01-01 07:00:00|1        |300          |
|2025-01-01 07:00:00|2025-01-01 07:05:00|1        |300          |
|2025-01-01 07:05:00|2025-01-01 07:10:00|1        |300          |
|2025-01-0

In [3]:
from pyspark.sql.window import Window

w = Window.partitionBy("trayectoriaId").orderBy("eventTimestamp")

pares = (
    df
    .filter(F.col("velocity").isNotNull())
    .withColumn(
        "velocidad_siguiente",
        F.lead("velocity", 1).over(w)
    )
    .withColumn(
        "fecha_siguiente",
        F.lead("eventTimestamp", 1).over(w)
    )
    .withColumn(
        "segundos_siguiente",
        F.unix_timestamp("fecha_siguiente")
        - F.unix_timestamp("eventTimestamp")
    )
    .filter(F.col("segundos_siguiente").between(1, 60))
)

print("PARES VALIDOS:", pares.count())

[Stage 31:>                                                         (0 + 2) / 2]

PARES VALIDOS: 11248662


In [4]:
from pyspark.sql import functions as F

ventanas.groupBy("aeronaves") \
    .count() \
    .orderBy("aeronaves") \
    .show(30, False)

ventanas.agg(
    F.min("aeronaves").alias("minimo"),
    F.max("aeronaves").alias("maximo"),
    F.avg("aeronaves").alias("promedio")
).show()

+---------+-----+
|aeronaves|count|
+---------+-----+
|1        |37739|
|2        |1098 |
|3        |11   |
+---------+-----+



[Stage 45:======================================================> (48 + 1) / 49]

+------+------+-----------------+
|minimo|maximo|         promedio|
+------+------+-----------------+
|     1|     3|1.028830313014827|
+------+------+-----------------+

